# Set Up

In [1]:
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.metricas_avaliacao import avaliacao_completa, metricas_juntando

In [2]:
df, metadados, variaveis = dados(r"data\processed\dados_med_tratados.xlsx")

# Experimento A
Base tratada (`dados_med_tratados.xlsx`: médias mensais, outliers tratados). A referência são os meses completos. Em cada rodada, valores são removidos ao acaso e imputados: média e mediana com 5% de falhas e KNN com 30% (a faixa em que o pipeline aplica cada método). São 30 rodadas (seeds de 0 a 29).

In [3]:
# tabela: uma linha por seed × método × variável (as métricas de cada rodada)
# celulas: uma linha por célula avaliada (valor verdadeiro e imputado), usada para juntar as rodadas
tabela, celulas = avaliacao_completa(df, n_seeds=30, devolver_celulas=True)
tabela.head()

,variavel,rmse,mae,r2,n_celulas,metodo,pct_faltantes,seed
0,COR,69.890878,64.212389,-0.005868,5,media,0.05,0
1,TURB.,53.223806,45.874336,-0.002340,5,media,0.05,0
2,pH,0.150585,0.122478,-1.180371,5,media,0.05,0
3,ALC.,14.586900,14.263717,-0.996787,5,media,0.05,0
4,AC.,3.942513,3.113274,-0.407918,5,media,0.05,0


Conferência: meses de referência, número de rodadas e células avaliadas por variável em cada rodada

In [4]:
print("Meses completos (referência):", len(df.dropna()))
print("Rodadas:", tabela["seed"].nunique())

# Células avaliadas por variável em cada rodada. O número é o mesmo em todas as rodadas,
# por isso basta a primeira linha de cada método (first)
tabela.groupby("metodo", sort=False)[["pct_faltantes", "n_celulas"]].first()

Meses completos (referência): 118
Rodadas: 30


,pct_faltantes,n_celulas
metodo,,
media,0.05,5
mediana,0.05,5
knn,0.30,35


# Métricas juntando as rodadas
RMSE, MAE e R² calculados uma vez sobre todas as células de cada método × variável (`metricas_juntando`). O DP mostra quanto o RMSE e o MAE variam de uma rodada para outra. O R² fica sem DP: com 5 células por rodada na média e na mediana, o R² de uma rodada oscila demais (é o motivo de juntar as rodadas).

In [5]:
juntando = metricas_juntando(celulas)

# DP entre as rodadas: quanto o RMSE e o MAE variam de uma rodada (seed) para outra.
# O groupby separa a tabela por método e variável; o std calcula o DP das 30 rodadas de cada grupo
desvio = tabela.groupby(["metodo", "variavel"], sort=False)[["rmse", "mae"]].std()

# add_suffix renomeia as colunas (rmse -> rmse_dp, mae -> mae_dp).
# reset_index devolve "metodo" e "variavel", que o groupby transformou em índice, para colunas comuns
desvio = desvio.add_suffix("_dp").reset_index()

# merge junta as duas tabelas pelas linhas com o mesmo método e a mesma variável
resumo = juntando.merge(desvio, on=["metodo", "variavel"])
resumo = resumo[["metodo", "pct_faltantes", "variavel", "rmse", "rmse_dp", "mae", "mae_dp",
                 "r2", "n_celulas", "n_seeds"]]
resumo

,metodo,pct_faltantes,variavel,rmse,rmse_dp,mae,mae_dp,r2,n_celulas,n_seeds
0,media,0.05,COR,85.262182,32.729295,67.398820,23.477550,-0.019924,150,30
1,media,0.05,TURB.,70.938156,24.046574,56.683068,16.965179,-0.016209,150,30
2,media,0.05,pH,0.153302,0.052803,0.122301,0.044843,-0.027537,150,30
3,media,0.05,ALC.,18.989151,5.387964,16.134277,4.822825,-0.026134,150,30
4,media,0.05,AC.,5.191109,2.428368,3.643304,1.731324,-0.018567,150,30
5,media,0.05,O.C.,1.830392,0.634564,1.467245,0.571128,-0.029649,150,30
6,media,0.05,O.D.,0.821624,0.299318,0.653180,0.274922,-0.043342,150,30
7,media,0.05,Cl,13.828824,5.780867,10.440590,4.489707,-0.018895,150,30
8,media,0.05,DUR.,14.832939,4.968159,12.102537,4.041280,-0.018881,150,30
9,media,0.05,Fe,1.655131,0.636262,1.330061,0.515725,-0.027120,150,30


Uma tabela por métrica: variáveis nas linhas; para cada método, o valor juntando as rodadas e, ao lado, o DP entre elas (só RMSE e MAE).

In [6]:
metodos = ["media", "mediana", "knn"]

for metrica in ["rmse", "mae", "r2"]:
    # pivot reorganiza o resumo: variáveis nas linhas, métodos nas colunas
    valor = resumo.pivot(index="variavel", columns="metodo", values=metrica)
    if metrica != "r2":
        dp = resumo.pivot(index="variavel", columns="metodo", values=f"{metrica}_dp")

    # Variáveis na ordem da base. As colunas são preenchidas pelo índice (nome da variável)
    tabela_metrica = pd.DataFrame(index=variaveis)
    for metodo in metodos:
        tabela_metrica[metodo] = valor[metodo]
        # O R² não tem DP (ver acima)
        if metrica != "r2":
            tabela_metrica[f"{metodo} (DP)"] = dp[metodo]

    # display mostra a tabela formatada, como a última linha de uma célula, mesmo dentro do laço
    print(metrica.upper())
    display(tabela_metrica.round(3))

RMSE


,media,media (DP),mediana,mediana (DP),knn,knn (DP)
COR,85.262,32.729,92.787,43.592,107.922,30.578
TURB.,70.938,24.047,80.094,32.001,71.656,9.370
pH,0.153,0.053,0.158,0.057,0.175,0.022
ALC.,18.989,5.388,18.799,5.828,17.920,2.771
AC.,5.191,2.428,5.497,2.848,6.165,1.292
O.C.,1.830,0.635,1.831,0.637,2.746,0.751
O.D.,0.822,0.299,0.821,0.295,0.904,0.113
Cl,13.829,5.781,13.781,6.078,13.522,2.245
DUR.,14.833,4.968,15.195,5.361,14.883,2.202
Fe,1.655,0.636,1.768,0.878,1.644,0.290


MAE


,media,media (DP),mediana,mediana (DP),knn,knn (DP)
COR,67.399,23.478,63.867,29.987,67.733,17.378
TURB.,56.683,16.965,53.620,21.644,49.486,8.601
pH,0.122,0.045,0.119,0.050,0.138,0.021
ALC.,16.134,4.823,15.580,5.220,13.525,2.064
AC.,3.643,1.731,3.460,1.898,4.370,1.024
O.C.,1.467,0.571,1.465,0.577,1.960,0.479
O.D.,0.653,0.275,0.649,0.273,0.716,0.099
Cl,10.441,4.490,10.187,4.763,9.921,1.712
DUR.,12.103,4.041,12.213,4.121,11.362,1.656
Fe,1.330,0.516,1.219,0.664,1.098,0.207


R2


,media,mediana,knn
COR,-0.020,-0.208,-0.107
TURB.,-0.016,-0.295,-0.053
pH,-0.028,-0.087,-0.267
ALC.,-0.026,-0.006,0.118
AC.,-0.019,-0.142,-0.489
O.C.,-0.030,-0.031,-0.424
O.D.,-0.043,-0.043,-0.200
Cl,-0.019,-0.012,0.005
DUR.,-0.019,-0.069,-0.041
Fe,-0.027,-0.172,0.220


Gravação dos resultados: aba `resumo` (métricas juntando + DP do RMSE e do MAE) e aba `rodadas` (uma linha por seed × método × variável). Como as seeds são fixas, rodar de novo dá os mesmos números.

In [7]:
caminho = BASE_DIR / "data" / "output" / "avaliacao_imputacoes_A.xlsx"

# ExcelWriter permite gravar várias abas no mesmo arquivo
with pd.ExcelWriter(caminho) as arquivo:
    resumo.to_excel(arquivo, sheet_name="resumo", index=False)
    tabela.to_excel(arquivo, sheet_name="rodadas", index=False)

print("Gravado em:", caminho)

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2\data\output\avaliacao_imputacoes_A.xlsx
